# Preprocessing — Telugu-English code-mixed corpus

The cleaning logic now lives in `src/preprocessing.py` so it can be unit
tested. This notebook is for exploring a corpus, not for defining functions.

Two bugs from the earlier version of this notebook are fixed there:

1. `CleanText` stripped `[^\x00-\x7F]`, which deleted **every Telugu
   character** — the entire point of a code-mixed corpus.
2. `CleanText(files_path, output_file, cleaned_output_file)` took three file
   paths but was called as `df["text"].apply(CleanText)`, passing one string,
   so it raised `TypeError`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd

from preprocessing import clean_text, script_of, tokenize
from corpus import load_corpus, audit, format_report

## Load the corpus

In [ ]:
# Relative to the repo root, so this works on any machine.
DATA = Path("..") / "data" / "Raw"

records = []
for path in sorted(DATA.glob("*.json")):
    records.extend(load_corpus(path))

df = pd.DataFrame(records)
print(f"{len(df):,} records")
df.head()

## Audit before anything else

Check the corpus is real language before spending time on it. The bundled
`data/` files fail this check: a 36-word vocabulary with near-uniform token
frequencies, which is random sampling rather than natural text.

In [ ]:
print(format_report(audit(records, sample=3000)))

## Clean the text

`clean_text` is a single-argument pure function, so `.apply` works.

In [ ]:
df["clean"] = df["text"].apply(clean_text)
df[["text", "clean"]].head()

In [ ]:
# Telugu must survive cleaning — this is the regression that mattered.
sample = "నేను office కి వెళ్తున్నాను"
print(clean_text(sample))
print([(t, script_of(t)) for t in tokenize(clean_text(sample))])

## Script distribution

In [ ]:
from collections import Counter

scripts = Counter(
    script_of(tok)
    for text in df["clean"].head(5000)
    for tok in tokenize(text)
)
print(scripts)